In [ ]:
#import frequently used libraries

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

from scipy.stats import norm
from scipy.stats import lognorm
from scipy.stats import chi2
from scipy.stats import t
from scipy.stats import f
from scipy.stats import skew
from scipy.stats import probplot

import scipy.stats as sc 
import statsmodels.graphics.gofplots as sm 

import sys
!{sys.executable} -m pip install stemgraphic
import stemgraphic

import seaborn as sns


In [ ]:
#Read the data from an Excel file

df=pd.read_excel("CallCenterData.xlsx")

print(df) #shows the content of the file or, if the file is large, the first and last 5 rows of the file

print(df.columns) #shows the names of the columns, which you will use to refer 

#Create separate columns from the dataframe
period = df["Period"]
day = df["Day"]
answered = df["Answered"]
made = df["Made"]

In [ ]:
#Processing the data frame
#Finding the ratio of two columns or the sum of several columns
#Storing the resulting quantities as a new column in the data frame

#Ratio as success rate in Manual or Automated Calls
df['Proportion']=df['columnname1']/df['columnname2']
proportion = df["Proportion"]

#Sum of three columns - Sum is a new column below
df['Sum']=df['columnname1']+df['columnname2']+df['columnname3']

#Finding the sum of a column
Sum_columnname1=df['columnname1'].sum()

#OR with an array
arrayname=df['columnname1']
Sum=arrayname.sum()


In [ ]:
#Display descriptive statistics

#Desciptive stats for each column (excpet for skewness)
df.describe()

#Function for all descriptive stats
def descriptive_stats(column, column_name):
    sample=column_name
    sample_mean=np.mean(column)
    sample_std=np.std(column, ddof=1)
    sample_min=np.min(column)
    sample_max=np.max(column)
    quartile_one=np.quantile(column,.25)
    quartile_three=np.quantile(column,.75)
    med_ian= np.median(column)
    skewness=column.skew(axis = 0, skipna = True) 
    
    return sample,sample_mean,sample_std,sample_min, sample_max, quartile_one, med_ian, quartile_three,skewness

#Print all descriptive stats in good format

print("      Sample", "    Mean", " Std Dev", "     Min", "     Max", " 1st Quartile", "  Median", " 3rd Quartile", " Skewness")
print("%12s %8.4f %8.4f %8.4f %8.4f %13.4f %8.4f %13.4f %9.4f" % descriptive_stats(df[column_name],column_name))



In [ ]:
#define a function to get histogram of each column
def plot_histo(cstr, column):
    plt.hist(column)
    plt.title(cstr)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.show()
    
    
    # Add title and axis names



In [ ]:
#Removing the outliers
Q1a = column.quantile(0.25)
Q3a = column.quantile(0.75)

IQRa = Q3a - Q1a

lower_bounda = Q1a - 1.5 * IQRa
upper_bounda = Q3a + 1.5 * IQRa

column_no_outliers = df[
    (column >= lower_bounda) & (column <= upper_bounda)]

In [ ]:
#Plotting all boxplots side by side
n = 4

plt.figure(figsize=(15, 5)) #figure size for better visibility
plt.subplot(1, n, 1) #1 row <# columns> columns 1st plot
plt.boxplot(column1)
plt.title('columnname1')

plt.subplot(1, n, 2)
plt.boxplot(column2)
plt.title('columnname2')

plt.subplot(1, n, 3)
plt.boxplot(column3)
plt.title('columnname3')


In [ ]:
#Plotting the normal probability plots
plt.figure(figsize=(10, 10))
sm.ProbPlot(column1, sc.norm, fit=True).ppplot(line='45') #change column name here for all columns
plt.title('columnname1 vs normal distribution')


In [ ]:
#Plotting the data with respect to observation sequence or time - time series plot
# plt.plot(column) #plot of column name with respect to time

plt.plot(column1) #plot of Answered calls with respect to time




In [ ]:
data = [column1, column2]

# Create boxplots
plt.boxplot(data, labels=['columnname1', 'columnname2'])

# Add title and labels
plt.title('Comparable Boxplots')
plt.xlabel(' ')
plt.ylabel('YYY')

# Show plot
plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(column1, column2)


plt.xlabel("Period")
plt.ylabel("YYY")
plt.title("TTT")
plt.legend()

plt.show()

In [ ]:
#Finding the CI for the population mean
#Normal with known population standard deviation labeled as sigma

def ci_mean_knownvar(column,sigma,alpha):
    
    z_score = norm.ppf(1-alpha/2)
    
    sample_mean=np.mean(column)
    sample_cnt=len(column)
    
    lcl = sample_mean - z_score* (sigma/((sample_cnt)**0.5))
    
    ucl = sample_mean + z_score* (sigma/(sample_cnt)**0.5)
    
    return lcl,ucl

In [ ]:
#Finding the CI for the population mean
#Large sample with unknown population standard deviation

def ci_mean_large(column,alpha):
    
    z_score = norm.ppf(1-alpha/2)
    
    sample_mean=np.mean(column)
    sample_std=np.std(column, ddof=1) #to find sample standard deviation we need to set ddof=1, that corresponds to (n-1)
    sample_cnt=len(column)
    
    lcl = sample_mean - z_score* (sample_std/((sample_cnt)**0.5))
    
    ucl = sample_mean + z_score* (sample_std/(sample_cnt)**0.5)
    
    return lcl,ucl

In [ ]:
#Finding the CI for the population mean
#Small normal samples with unknown population standard deviation

def ci_mean_smallnorm(column,alpha):
    
    sample_cnt=len(column)
    
    t_score = t.ppf(1-alpha/2,sample_cnt-1)
    
    sample_mean=np.mean(column)
    sample_std=np.std(column, ddof=1) #to find sample standard deviation we need to set ddof=1, that corresponds to (n-1)
        
    lcl = sample_mean - t_score* (sample_std/((sample_cnt)**0.5))
    
    ucl = sample_mean + t_score* (sample_std/(sample_cnt)**0.5)
    
    return lcl,ucl

In [ ]:
#Finding the CI for the difference between the two population means
#Normal with known population standard deviations labeled as sigma1 and sigma2

def diff_ci_mean_knownvar(column1,column2,sigma1,sigma2,alpha):
    
    z_score = norm.ppf(1-alpha/2)
    
    sample_mean1=np.mean(column1)
    sample_mean2=np.mean(column2)

    sample_cnt1=len(column1) 
    sample_cnt2=len(column2)
    
    dif_std=(sigma1**2/sample_cnt1+sigma2**2/sample_cnt2)**0.5
    
    lcl = sample_mean1-sample_mean2 - z_score* dif_std
    
    ucl = sample_mean1-sample_mean2  + z_score* dif_std
    
    return lcl,ucl

In [ ]:
#Finding the CI for the difference between the two population means
#Large samples with unknown population standard deviations 

def diff_ci_mean_large(column1,column2,alpha):
    
    z_score = norm.ppf(1-alpha/2)
    
    sample_mean1=np.mean(column1)
    sample_mean2=np.mean(column2)
    sample_std1=np.std(column1, ddof=1)
    sample_std2=np.std(column2, ddof=1)
    sample_cnt1=len(column1) 
    sample_cnt2=len(column2)
    
    dif_std=(sample_std1**2/sample_cnt1+sample_std2**2/sample_cnt2)**0.5
    
    lcl = sample_mean1-sample_mean2 - z_score* dif_std
    
    ucl = sample_mean1-sample_mean2  + z_score* dif_std
    
    return lcl,ucl

In [ ]:
#Finding the CI for the difference between the two population means
#Small normal samples with unknown population standard deviations 

def diff_ci_mean_smallnorm(column1,column2,alpha):
    
    sample_mean1=np.mean(column1)
    sample_mean2=np.mean(column2)
    sample_std1=np.std(column1, ddof=1)
    sample_std2=np.std(column2, ddof=1)
    sample_cnt1=len(column1) 
    sample_cnt2=len(column2)

    #compute the degrees of freedom
    t_degree = (sample_std1**2/sample_cnt1+sample_std2**2/sample_cnt2)**2/(((sample_std1**2/sample_cnt1)**2)/(sample_cnt1-1)+((sample_std2**2/sample_cnt2)**2)/(sample_cnt2-1))
    t_degree = math.floor(t_degree)
    t_score = t.ppf(1-alpha/2,t_degree)

    
    dif_std=(sample_std1**2/sample_cnt1+sample_std2**2/sample_cnt2)**0.5
    
    lcl = sample_mean1-sample_mean2 - t_score* dif_std
    
    ucl = sample_mean1-sample_mean2  + t_score* dif_std
    
    return lcl,ucl

In [ ]:
#Finding the CI for the population proportion in the new style
#n: total number of trials; X: total number of successes

def ci_proportion(X,n, alpha):
    
    z_score = norm.ppf(1-alpha/2)
    
    p_new=(X+2)/(n+4)
    std_p_new=((p_new*(1-p_new))/(n+4))**(0.5)
    
    lcl = max(0,p_new - z_score*std_p_new)
    
    ucl = p_new + z_score*std_p_new
    
    return lcl,ucl

In [ ]:
#Finding the CI for the difference in the two population proportions in the new style
#nX: total number of trials; X: total number of successes - population 1
#nY: total number of trials; Y: total number of successes - population 2

def ci_dif_prop(X,nX,Y,nY, alpha):
    
    z_score = norm.ppf(1-alpha/2)
    
    pX_new=(X+1)/(nX+2)
    pY_new=(Y+1)/(nY+2)
    
    std_p_new=((pX_new*(1-pX_new))/(nX+2)+(pY_new*(1-pY_new))/(nY+2))**(0.5)
    
    lcl = pX_new -pY_new - z_score*std_p_new
    
    ucl = pX_new -pY_new + z_score*std_p_new
    
    return lcl,ucl